# Databricks, Delta & Data Lake

## 1. Databricks Workspace

Databricks Workspace is the environment used to organize and work with notebooks, files, folders, and other project resources.

### Main components

- **Workspace** — Organizes notebooks, folders, and workspace resources.
- **Compute** — Provides the processing resources required to run notebooks and workloads.
- **Files** — Used to work with files available in the workspace.
- **Volumes** — Used to store and access data files within the Unity Catalog structure.
- **Serverless** — Databricks-managed compute that can be used without manually managing the underlying infrastructure.


### Workspace Structure

A typical Databricks workflow can be represented as:

Workspace
↓
Notebook
↓
Compute
↓
Data
↓
Analysis / Processing

## 2. Catalog, Schema & Tables

Databricks uses a three-level namespace to organize data:

catalog.schema.table

### Components

- **Catalog** — Top-level container for organizing schemas.
- **Schema** — Organizes tables, views, and other data objects within a catalog.
- **Table** — Stores structured data.

### Example

workspace.default.titanic_processed

- `workspace` → Catalog
- `default` → Schema
- `titanic_processed` → Table

### Managed vs External Tables

**Managed Table**
- Databricks manages the table metadata and underlying data storage.

**External Table**
- Databricks manages the table metadata, while the underlying data remains in an externally specified storage location.

### Three-Level Naming

The general format is:

catalog.schema.table

Example:

workspace.default.titanic_processed

### Checking Catalogs and Schemas

Databricks SQL can be used to explore the available catalogs and schemas.

In [0]:
%sql
SHOW CATALOGS;

catalog
samples
system
workspace


In [0]:
%sql
SHOW SCHEMAS IN workspace;

databaseName
default
information_schema


In [0]:
%sql
SHOW TABLES IN workspace.default;

database,tableName,isTemporary
default,titanic_clean,false
default,titanic_encoded,false
default,titanic_processed,false


### Three-Level Table Reference

Tables can be referenced using:

catalog.schema.table

Example:

workspace.default.titanic_processed

In [0]:
%sql
SELECT *
FROM workspace.default.titanic_processed
LIMIT 5;

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Cabin_Deck
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,S,Unknown
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,S,Unknown
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,S,C
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,S,Unknown


## 3. SQL Editor

Databricks SQL Editor is used to query and analyze data using SQL.

Common SQL operations include:

- Selecting and filtering data
- Aggregating data
- Grouping and sorting data
- Joining tables
- Using subqueries and CTEs
- Using window functions

In [0]:
%sql
SELECT 
    Pclass,
    COUNT(*) AS passenger_count
FROM workspace.default.titanic_processed
GROUP BY Pclass
HAVING COUNT(*) > 300;

Pclass,passenger_count
3,491


In [0]:
%sql
SELECT 
    Sex,
    AVG(Age) AS average_age
FROM workspace.default.titanic_processed
GROUP BY Sex;

Sex,average_age
male,30.14067590987868
female,27.929936305732483


### Key Concepts

`WHERE` filters individual rows before grouping.

`GROUP BY` creates groups based on one or more columns.

`HAVING` filters groups after aggregation.

Window functions perform calculations across related rows while retaining individual rows.

## 4. Kaggle → Databricks

Datasets can be downloaded from Kaggle and uploaded to Databricks for analysis and processing.

### Workflow

Kaggle
↓

Download dataset
↓

Upload dataset to Databricks
↓

Store the file in a Volume
↓

Access the file using its Volume path
↓

Read the dataset into a DataFrame

In [0]:
import pandas as pd

df = pd.read_csv("/Volumes/workspace/default/ml_data/Titanic-Dataset.csv")

### Volume Path Structure

The general Volume path follows:

/Volumes/<catalog>/<schema>/<volume>/<file>

Example:

/Volumes/workspace/default/ml_data/Titanic-Dataset.csv

- `workspace` → Catalog
- `default` → Schema
- `ml_data` → Volume
- `Titanic-Dataset.csv` → File

A Volume is used to store and access files within the Unity Catalog structure. It is different from a Workspace location, which is primarily used for notebooks and workspace resources.

## 5. Delta Tables

Delta Lake is a storage layer used with Databricks that provides reliability and data-management features on top of Parquet data.

### Delta Table Structure

A Delta table consists of:

- Parquet data files — store the actual data.
- `_delta_log` — stores transaction and table-state information.

Conceptually:

Delta Table

├── Parquet data files

└── _delta_log

### Parquet vs Delta

**Parquet**
- File-based columnar storage format.
- Stores data efficiently.
- Does not provide Delta's transaction-management features.

**Delta**
- Uses Parquet data files.
- Maintains a `_delta_log`.
- Supports ACID transactions, table history, and time travel.

### ACID Transactions

Delta provides ACID transaction guarantees:

- **Atomicity** — A transaction is completed fully or rolled back.
- **Consistency** — Data remains in a valid state according to defined rules.
- **Isolation** — Concurrent transactions do not improperly interfere with each other.
- **Durability** — Successfully committed changes persist.

In [0]:
%sql
DESCRIBE DETAIL workspace.default.titanic_processed;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6a20e0be-797c-426b-a3a2-d52ddfd268fb,workspace.default.titanic_processed,null,,2026-09-21T11:40:25.524Z,2026-09-28T09:29:15.000Z,List(),List(),1,23183,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## 6. Save Preprocessed Data

Processed data can be saved as a table so that it can be accessed and reused later.

The `saveAsTable()` method is used to save a Spark DataFrame as a table.

### Basic Syntax

df.write.saveAsTable("catalog.schema.table")

In [0]:
spark_df_processed = spark.createDataFrame(df)
spark_df_processed.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.titanic_clean"
)

### Write Modes

Different write modes control what happens when the target table already exists:

- `error` / `errorifexists` — Raises an error if the table exists.
- `overwrite` — Replaces the existing data.
- `append` — Adds new data to the existing table.
- `ignore` — Does nothing if the table already exists.

### Important

`saveAsTable()` persists the DataFrame as a table in the specified catalog and schema.

The write mode determines how existing table data is handled.

## 7. Table Versions

Delta tables maintain different versions as transactions are committed.

Each successful transaction can create a new table version.

### Example

Version 0 → Initial table
Version 1 → Data appended
Version 2 → Data updated
Version 3 → Data deleted

A version number represents a table state after a committed transaction. It does not represent the number of rows in the table.

### Important

Different operations can create new table versions, including:

- INSERT / WRITE
- APPEND
- UPDATE
- DELETE
- MERGE
- OVERWRITE

For example:

Version 4
↓
UPDATE transaction
↓
Version 5

In [0]:
%sql
DESCRIBE HISTORY workspace.default.titanic_processed;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-09-28T09:29:15.000Z,76903904486299,sumeet14022003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1544323217464356),acfa57e1-4675-4f8f-9a77-383d047886c7,0928-085057-7hfyv4kr-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 23183, numDeletionVectorsRemoved -> 0, numOutputRows -> 891, numOutputBytes -> 23183)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
5,2026-09-23T12:00:10.000Z,76903904486299,sumeet14022003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1544323217464356),0455e65c-073b-4ae5-804d-ecfa9ccae85e,0923-112301-8uda7qrv-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 23183, numDeletionVectorsRemoved -> 0, numOutputRows -> 891, numOutputBytes -> 23183)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
4,2026-09-23T09:42:26.000Z,76903904486299,sumeet14022003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1544323217464356),862245c6-9caf-4b74-8305-2c89309a3783,0923-091041-rd4ge6c7-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 23183, numDeletionVectorsRemoved -> 0, numOutputRows -> 891, numOutputBytes -> 23183)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
3,2026-09-22T11:34:29.000Z,76903904486299,sumeet14022003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1544323217464356),46c59ef8-8c46-403d-89d7-813ba91d0294,0922-112806-4shk5694-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 23183, numDeletionVectorsRemoved -> 0, numOutputRows -> 891, numOutputBytes -> 23183)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
2,2026-09-22T05:03:43.000Z,76903904486299,sumeet14022003@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1544323217464356),987531eb-5b5d-4fa1-a79f-7fce5a1673b7,0922-050057-ovdq1pc4-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 23183, numDeletionVectorsRemoved -> 0, numOutputRows -> 891, numOutputBytes -> 23183)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.

The table history can be used to identify the version number, timestamp, operation, and other details associated with each transaction.

## 8. Table History

Delta provides table history to track changes made to a table.

### Command

DESCRIBE HISTORY workspace.default.titanic_processed;

The history records information about transactions and table versions.

### Important Columns

- `version` — Identifies the table version.
- `timestamp` — Shows when the transaction occurred.
- `operation` — Shows the operation performed, such as WRITE, UPDATE, or DELETE.
- `operationParameters` — Provides details about the operation.
- `operationMetrics` — Provides measurable results of the operation.

### Example

Version 5
Operation: UPDATE
Operation Metrics: Number of records affected

This means Version 5 represents the table state after the UPDATE transaction was committed.

Table history helps understand what changes occurred, when they occurred, and which table version was created by each transaction.

## 9. Time Travel

Time Travel allows us to read a previous version of a Delta table without changing the current table.

There are two common ways to access historical data:

1. Version-based Time Travel
2. Timestamp-based Time Travel

In [0]:
%sql
SELECT *
FROM workspace.default.titanic_processed
VERSION AS OF 2;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6885280454873199>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT *\nFROM workspace.default.titanic_processed\nVERSION AS OF 2;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:214, in SqlMagic.sql(self, line, cell)
    207 except BaseException as e:
    208  

### Version-based Time Travel

`VERSION AS OF` is used when the required table version is known.

The query reads the table as it existed at that version. It does not change the current table.

In [0]:
%sql
SELECT *
FROM workspace.default.titanic_processed
TIMESTAMP AS OF '2026-09-22 05:02:00';

### Timestamp-based Time Travel

`TIMESTAMP AS OF` is used when the required historical timestamp is known.

The query returns the table state corresponding to that time without modifying the current table.

### Time Travel vs Restore

**Time Travel**
- Reads or views a historical table state.
- Does not modify the current table.

**Restore**
- Makes a previous table state the current state.
- The restore itself creates a new table version.

## 10. Data Lake

A Data Lake is a centralized storage environment that can store large amounts of data from different sources and in different formats.

Data can come from:

- CSV and Excel files
- Databases
- APIs
- Application logs
- JSON and Parquet files
- Other data sources

### Medallion Architecture

Data is commonly organized into three logical layers:

🥉 Bronze → Raw data
🥈 Silver → Cleaned and transformed data
🥇 Gold → Business and analysis-ready data

### Bronze Layer

The Bronze layer contains raw or minimally processed data.

Example:

A raw Titanic CSV downloaded from Kaggle can be stored in the Bronze layer.

### Silver Layer

The Silver layer contains cleaned and transformed data.

Examples:

- Handling missing values
- Cleaning data
- Removing unnecessary columns
- Creating derived columns
- Transforming data types

### Gold Layer

The Gold layer contains business or analysis-ready data.

Examples:

- Aggregated results
- Business metrics
- Analytical datasets
- Dashboard-ready data

### Delta and Data Lake

Delta tables can be used within the different layers of a Data Lake.

Bronze → Delta table
Silver → Delta table
Gold → Delta table

Delta provides features such as ACID transactions, table history, and time travel, while the Bronze, Silver, and Gold layers organize data according to its processing stage.